# CNN heat-model training (Colab)

Trains the Keras/TensorFlow CNN heat model (S5/C2) -- predicts `lst_bicubic10` from
Sentinel-2 + spectral-index + land-cover patches -- on a GPU. Run this from
**VS Code**: `Select Kernel` -> `Colab` -> sign in -> `New Colab Server` ->
GPU (free T4 is enough) -> connect.

**Prerequisite**: this notebook needs the land-cover hybrid raster, which
can only be produced locally (U-Net inference + RF combined via
`scripts/build_landcover_hybrid.py`) -- Colab can't regenerate it. Run
`train_unet.ipynb` and the local U-Net inference steps first, then push the
resulting hybrid raster once (and after every relabel/retrain):

```bash
python -c "from src.utils import gcs; from config.settings import GCS_MODEL_BUCKET, HYBRID_RASTER_GCS_PREFIX; gcs.upload_file('data/processed/landcover/hybrid_landcover.tif', GCS_MODEL_BUCKET, f'{HYBRID_RASTER_GCS_PREFIX}.tif')"
```

**One-time setup**: none needed in advance -- same as `train_unet.ipynb`, the
auth cell below prompts a masked paste-in box (`getpass`) for your
service-account JSON key each session.

In [9]:
# --- Repo sync (same pattern as train_unet.ipynb) --------------------------
# Colab's /content disk is empty every session -- clone (or pull, if this
# session already has it) the public repo so `src/`/`config/` are importable
# the same "sys.path manipulation, no packaging" way every local script uses.
import os
import subprocess
import sys

REPO_URL = "https://github.com/EngineerKX/urban-heat-cooling-priority.git"
REPO_DIR = "/content/urban-heat-cooling-priority"
# Branch to train from. Colab clones the repo's default branch (main) unless
# told otherwise -- set this to run against in-progress work on a feature
# branch instead. Must already be PUSHED to the remote; a purely local commit
# is invisible here. kxbranch carries the 300-point hand-labeled validation
# sample (0afe082/15c2935/eb35be7) plus the TF/Keras architecture -- the old
# "sherwin-setup" pointer predates the hand-labeled data entirely.
REPO_BRANCH = "kxbranch"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )
else:
    # Existing session: make sure we're on REPO_BRANCH and up to date, not
    # still on whatever was cloned earlier.
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(
        ["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR, check=True
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo ready at", REPO_DIR, "on branch", REPO_BRANCH)

Repo ready at /content/urban-heat-cooling-priority on branch kxbranch


In [10]:
%pip install -q -r {REPO_DIR}/requirements-colab.txt


In [11]:
# --- Auth: service account, not interactive Google login -------------------
# Paste the full contents of your service-account JSON key when prompted
# below (open credentials/nus-iss-urban-heat-sg-*.json in a text editor,
# copy everything, Ctrl+V into the box that appears). Uses getpass -- a
# plain stdin prompt, standard Jupyter protocol -- not a browser widget,
# since google.colab.files.upload() hung indefinitely through VS Code's
# remote-kernel connection to Colab (confirmed 2026-08-05). getpass also
# never echoes the input back into the cell's saved output, unlike a plain
# input() -- this repo is public on GitHub, so that distinction matters.
import getpass
import os

key_json = getpass.getpass("Paste the full contents of your service-account JSON key, then press Enter: ")

key_path = "/content/gee_key.json"
with open(key_path, "w") as f:
    f.write(key_json)

os.environ["GEE_PRIVATE_KEY_PATH"] = key_path
os.environ["GEE_SERVICE_ACCOUNT"] = "urban-heat-pipeline@nus-iss-urban-heat-sg.iam.gserviceaccount.com"
os.environ["GEE_PROJECT_ID"] = "nus-iss-urban-heat-sg"
os.environ["GEE_EXPORT_BUCKET"] = "nus-iss-urban-heat-sg-exports"
print("Service-account credentials staged.")

Service-account credentials staged.


In [ ]:
import json
import time
from pathlib import Path

import ee
import mlflow
import tensorflow as tf

from config import settings
from config.settings import (
    CNN_MODEL_SAVE_PATH,
    DRY_SEASON_MONTHS,
    GCS_MODEL_BUCKET,
    GEE_EXPORT_BUCKET,
    HYBRID_RASTER_GCS_PREFIX,
    LANDSAT_CLOUD_COVER_MAX,
    NATIVE_SCALE_M,
    S2_CLOUD_PROB_MAX,
    S2_UTM_CRS,
    SG_BBOX,
    TARGET_SCALE_M,
    YEARS,
)
from src.downscaling.variants import build_lst_30m, variant_bicubic10
from src.heat_model.cnn_data import build_local_feature_target_patches
from src.heat_model.cnn_train import train_cnn_regressor
from src.ingest.gee import export_geotiff_to_gcs, init_ee
from src.ingest.subzones import as_ee_feature_collection, dissolve_boundary, fetch_subzones_geojson
from src.landcover.rf_baseline import build_feature_image
from src.landcover.unet_data import INFERENCE_PATCH_DIR, export_inference_patches
from src.utils import gcs
from src.utils.experiment_tracking import HEAT_MODEL_EXPERIMENT_NAME, export_run_summary, start_run
from src.utils.seed import set_all_seeds

In [13]:
set_all_seeds()
init_ee()
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", bool(gpus), "-", gpus[0].name if gpus else "none")

GPU available: True - /physical_device:GPU:0


In [14]:
# --- Preview the model architecture (no training, no GPU needed) ----------
# Every Conv2D / MaxPooling2D / Conv2DTranspose / Concatenate layer, its
# output shape, and parameter count -- printed here so it's visible
# regardless of whether the training cell below ends up actually training
# (a cache hit skips straight past model.summary() inside
# train_cnn_regressor()). Handy for tuning CNN_BASE_FILTERS /
# UNET_PATCH_SIZE in config/settings.py before committing to a full
# training run: just re-run this cell after editing settings to see the
# new shapes/param counts immediately. Builds the graph only -- not
# compiled or trained. Same backbone as U-Net's preview cell, just with
# CNN's wider input (feature bands + one-hot land-cover channels) and a
# linear head instead of softmax.
from config.settings import ALL_FEATURE_BANDS, CNN_BASE_FILTERS, UNET_PATCH_SIZE
from src.heat_model.cnn_model import N_LANDCOVER_CLASSES, build_cnn_regressor

preview_in_channels = len(ALL_FEATURE_BANDS) + N_LANDCOVER_CLASSES
preview_model = build_cnn_regressor(
    (UNET_PATCH_SIZE, UNET_PATCH_SIZE, preview_in_channels), base_filters=CNN_BASE_FILTERS,
)
preview_model.summary()

Model: "heat_cnn_regressor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 13)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 128, 128,  │      3,776 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 128, 128,  │      9,248 │ conv2d_15[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ conv2d_16[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 64, 64,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 64, 64,    │     36,928 │ conv2d_17[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 32, 32,    │          0 │ conv2d_18[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 32, 32,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 32, 32,    │    147,584 │ conv2d_19[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 16, 16,    │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 16, 16,    │    295,168 │ max_pooling2d_5[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 16, 16,    │    590,080 │ conv2d_21[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_3  │ (None, 32, 32,    │    131,200 │ conv2d_22[0][0]   │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 32, 32,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_20[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │    295,040 │ concatenate_3[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │    147,584 │ conv2d_23[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_4  │ (None, 64, 64,    │     32,832 │ conv2d_24[0][0] 

 Total params: 1,928,481 (7.36 MB)

 Trainable params: 1,928,481 (7.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# --- Pull the hybrid raster (the one input Colab can't self-generate) ---
HYBRID_RASTER_PATH = Path("data/processed/landcover/hybrid_landcover.tif")
HYBRID_RASTER_PATH.parent.mkdir(parents=True, exist_ok=True)
gcs.download_blob(GCS_MODEL_BUCKET, f"{HYBRID_RASTER_GCS_PREFIX}.tif", HYBRID_RASTER_PATH)
print(f"Hybrid raster ready at {HYBRID_RASTER_PATH}")

In [16]:
# --- Build the same GEE feature image U-Net used (no labels needed here) --
sg_bbox = ee.Geometry.Rectangle(list(SG_BBOX))
subzones_fc = as_ee_feature_collection(fetch_subzones_geojson())
boundary = dissolve_boundary(subzones_fc)
feature_image, _valid_mask = build_feature_image(sg_bbox, boundary, YEARS, DRY_SEASON_MONTHS, S2_CLOUD_PROB_MAX)
print("Feature image built.")

Dissolved Singapore boundary built. Approx area: 788.3 km²
(Sanity check: Singapore's land area is ~730-735 km².)
Feature image built.


In [17]:
# --- Export (or reuse the GCS-cached) inference patches --------------------
# Same patches U-Net's local inference uses -- existence-cached in GCS, so
# this is a fast download if train_unet.ipynb or a local inference run
# already produced them, and a real (one-time) GEE export otherwise.
inference_patch_dir = export_inference_patches(feature_image, boundary)
mixer_json_path = inference_patch_dir / "unet_inference.json"

Inference patches already exist at gs://nus-iss-urban-heat-sg-exports/unet_inference_patches/unet_inference — downloading instead of re-exporting from GEE.


In [18]:
# --- Pull (or export) lst_bicubic10 -----------------------------------------
# The only new GEE computation S5's CNN half needs -- variant_bicubic10
# reused unmodified over the same season window as the production heat
# variants. Already sitting in GCS from earlier work, so this is normally
# just a download; the export branch exists for a from-scratch setup.
LST_BICUBIC10_GCS_PREFIX = "heat_model/lst_bicubic10_full"
LST_BICUBIC10_PATH = Path("data/interim/lst_bicubic10_full.tif")
LST_BICUBIC10_PATH.parent.mkdir(parents=True, exist_ok=True)

if gcs.blob_exists(GEE_EXPORT_BUCKET, f"{LST_BICUBIC10_GCS_PREFIX}.tif"):
    print("lst_bicubic10 already exported — downloading from GCS instead of re-running the GEE export.")
    gcs.download_blob(GEE_EXPORT_BUCKET, f"{LST_BICUBIC10_GCS_PREFIX}.tif", LST_BICUBIC10_PATH)
else:
    lst_30m = build_lst_30m(sg_bbox, YEARS, DRY_SEASON_MONTHS, LANDSAT_CLOUD_COVER_MAX, S2_UTM_CRS, NATIVE_SCALE_M)
    bicubic_image = variant_bicubic10(lst_30m, S2_UTM_CRS, TARGET_SCALE_M)
    export_geotiff_to_gcs(
        bicubic_image, description="lst_bicubic10_full", bucket=GEE_EXPORT_BUCKET,
        prefix=LST_BICUBIC10_GCS_PREFIX, region=sg_bbox, scale=TARGET_SCALE_M, crs=S2_UTM_CRS,
        out_path=LST_BICUBIC10_PATH,
    )

lst_bicubic10 already exported — downloading from GCS instead of re-running the GEE export.


In [ ]:
# --- Build (X, y, valid_mask) patches, channels-last (n, H, W, C) --------
X, y, valid_mask = build_local_feature_target_patches(
    inference_patch_dir, mixer_json_path, HYBRID_RASTER_PATH, LST_BICUBIC10_PATH,
)

In [20]:
# --- Train -------------------------------------------------------------
with start_run("cnn", experiment_name=HEAT_MODEL_EXPERIMENT_NAME):
    mlflow.log_params({
        "n_patches": int(X.shape[0]),
        "patch_size": int(X.shape[1]),
        "n_channels": int(X.shape[3]),
        "cnn_base_filters": settings.CNN_BASE_FILTERS,
        "cnn_epochs": settings.CNN_EPOCHS,
        "cnn_learning_rate": settings.CNN_LEARNING_RATE,
        "cnn_batch_size": settings.CNN_BATCH_SIZE,
        "cnn_early_stop_patience": settings.CNN_EARLY_STOP_PATIENCE,
        "force_retrain": False,
    })

    model, history = train_cnn_regressor(X, y, valid_mask, force_retrain=False)

    if history is not None:
        for epoch, (loss, train_metric, val_loss, val_metric) in enumerate(
            zip(history["train_loss"], history["train_metric"], history["val_loss"], history["val_metric"])
        ):
            mlflow.log_metrics(
                {"train_loss": loss, "train_rmse": train_metric, "val_loss": val_loss, "val_rmse": val_metric},
                step=epoch,
            )
        mlflow.log_metric("best_val_loss", min(history["val_loss"]))

print(f"Model saved to {CNN_MODEL_SAVE_PATH}")

2026/09/18 08:46:18 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/18 08:46:18 INFO mlflow.store.db.utils: Updating database tables
2026/09/18 08:46:20 INFO mlflow.tracking.fluent: Experiment with name 'heat_model_s5' does not exist. Creating a new experiment.


Train patches: 906, validation patches: 160


Model: "heat_cnn_regressor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 13)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 128, 128,  │      3,776 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 128, 128,  │      9,248 │ conv2d_30[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 64, 64,    │          0 │ conv2d_31[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 64, 64,    │     18,496 │ max_pooling2d_6[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 64, 64,    │     36,928 │ conv2d_32[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 32, 32,    │          0 │ conv2d_33[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 32, 32,    │     73,856 │ max_pooling2d_7[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 32, 32,    │    147,584 │ conv2d_34[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 16, 16,    │          0 │ conv2d_35[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_36 (Conv2D)  │ (None, 16, 16,    │    295,168 │ max_pooling2d_8[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_37 (Conv2D)  │ (None, 16, 16,    │    590,080 │ conv2d_36[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_6  │ (None, 32, 32,    │    131,200 │ conv2d_37[0][0]   │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 32, 32,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_35[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_38 (Conv2D)  │ (None, 32, 32,    │    295,040 │ concatenate_6[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_39 (Conv2D)  │ (None, 32, 32,    │    147,584 │ conv2d_38[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_7  │ (None, 64, 64,    │     32,832 │ conv2d_39[0][0] 

 Total params: 1,928,481 (7.36 MB)

 Trainable params: 1,928,481 (7.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 20s 72ms/step - loss: 974.0790 - rmse: 48.6315 - val_loss: 40.2732 - val_rmse: 10.0917
Epoch 2/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 33.7932 - rmse: 9.0581 - val_loss: 33.0132 - val_rmse: 9.1369
Epoch 3/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 29.3724 - rmse: 8.4448 - val_loss: 27.7926 - val_rmse: 8.3834
Epoch 4/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 26.8172 - rmse: 8.0691 - val_loss: 30.5573 - val_rmse: 8.7905
Epoch 5/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 24.6033 - rmse: 7.7289 - val_loss: 29.7229 - val_rmse: 8.6697
Epoch 6/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 19.6055 - rmse: 6.8994 - val_loss: 20.6409 - val_rmse: 7.2247
Epoch 7/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 20.2662 - rmse: 7.0147 - val_loss: 20.3484 - val_rmse: 7.1734
Epoch 8/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 17.0850 - rmse: 6.4406 - val_loss: 19.0614 - val_rmse: 6.9428
Epoch 9/30
1

In [21]:
# --- Push the trained weights + this run's MLflow summary to GCS ----------
if history is not None:
    summary = export_run_summary(
        "cnn", HEAT_MODEL_EXPERIMENT_NAME,
        params={
            "n_patches": int(X.shape[0]),
            "patch_size": int(X.shape[1]),
            "n_channels": int(X.shape[3]),
            "cnn_base_filters": settings.CNN_BASE_FILTERS,
            "cnn_epochs": settings.CNN_EPOCHS,
            "cnn_learning_rate": settings.CNN_LEARNING_RATE,
            "cnn_batch_size": settings.CNN_BATCH_SIZE,
            "cnn_early_stop_patience": settings.CNN_EARLY_STOP_PATIENCE,
            "force_retrain": False,
        },
        metrics_history={
            "train_loss": history["train_loss"],
            "train_rmse": history["train_metric"],
            "val_loss": history["val_loss"],
            "val_rmse": history["val_metric"],
        },
        final_metrics={"best_val_loss": min(history["val_loss"])},
    )
    gcs.upload_text(json.dumps(summary), GCS_MODEL_BUCKET, f"training_runs/cnn_{int(time.time())}.json")
    print("Run summary pushed for local MLflow import.")

!python {REPO_DIR}/scripts/push_models.py --model cnn

Run summary pushed for local MLflow import.
[cnn] Uploading /content/urban-heat-cooling-priority/models/heat_cnn.keras -> gs://nus-iss-urban-heat-sg-exports/models/heat_cnn.keras ...
[cnn] Pushed OK.


## Next steps (on your own machine, no GPU needed)

```bash
python scripts/pull_models.py --model cnn
```

Downloads the trained weights (verified via sha256) and imports this run into
your local MLflow store. From here, `scripts/diagnose_heat_model.py` and
`scripts/run_counterfactual.py` can run the CNN locally on CPU, and the
Streamlit app's Counterfactual Greening page runs it live.